# silver_carriers

Promotes the carriers reference table from bronze to silver. This is the simplest notebook in the silver layer and that's intentional. Carriers is a root dimension (20 rows, CSV-only, nothing FKs out of it), so there's no UNION work, no rejects to write, no business logic to apply. The interesting stuff starts next door with `silver_agents`.

## Why this notebook exists at all

Two reasons. One: every fact and dim table downstream eventually joins to carriers, so it has to exist in silver before anything else can. Two: building it first means the cross-lakehouse read pattern, the silver lineage stamp, and the schema-enforced write all get figured out on a 20-row table before we hit anything more painful.

## Source and output

Reads from `bronze_citadel_mga.bronze_carriers` (20 rows, CSV-sourced).
Writes to `silver_citadel_mga.silver_carriers`.

Three lineage columns get added on the way through:
- `_silver_load_timestamp`: when silver promoted this row
- `_silver_batch_id`: which silver run produced it
- `_bronze_batch_id`: carried forward from bronze so any silver row can be traced to its origin batch

## Design choices worth knowing

Schema enforcement is on. That's the opposite of bronze. Bronze accepts whatever the source gives us; silver decides what the contract looks like and enforces it. If carriers gains a column upstream, this notebook will fail loudly until someone updates it explicitly. That's the point.

Everything gets typed properly here. No string-stored dates or string-stored decimals making it into silver. Bronze can be sloppy; silver can't.

The whole table rebuilds on every run (`mode("overwrite")`). Carriers is small and idempotency matters more than incremental cleverness. If there's a bug, you re-run and the table is correct again.

## What's NOT here

No reject table. With 20 rows from a trusted source and no FKs going out, there's nothing meaningful to reject. The reject pattern shows up properly in `silver_agents`.

No SCD2. Carrier attributes (name, A.M. Best rating, state) do change in the real world, and a production version would track at least the rating as SCD2. We're deferring that to a v2. See `docs/dimensional-model/scd2-decisions.md` for which columns would qualify and why.

No business logic. Carriers is a reference table. It doesn't compute anything.


In [2]:
from pyspark.sql.functions import lit, current_timestamp, col
from pyspark.sql.types import StringType
import uuid
from datetime import datetime

# Batch ID for this run. Same convention as bronze, with a "silver_" prefix
# so we can tell silver-originated rows from bronze-originated ones when
# auditing _batch_id values downstream.
SILVER_BATCH_ID = f"silver_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

# Bronze read needs the three-part name because bronze is a referenced
# (non-default) lakehouse. Silver writes stay unqualified since silver
# is the default.
SOURCE_TABLE = "bronze_citadel_mga.dbo.bronze_carriers"
TARGET_TABLE = "silver_carriers"

print(f"Silver batch ID: {SILVER_BATCH_ID}")
print(f"Reading from:    {SOURCE_TABLE}")
print(f"Writing to:      {TARGET_TABLE}")

StatementMeta(, a9b72d9b-b798-41e3-930d-2e90eb28b94a, 4, Finished, Available, Finished, False)

Silver batch ID: silver_20260602_213851_0bd53d4f
Reading from:    bronze_citadel_mga.dbo.bronze_carriers
Writing to:      silver_carriers


In [3]:
# Read bronze_carriers and confirm what we're working with before promoting.
bronze_df = spark.read.table(SOURCE_TABLE)

print(f"Row count: {bronze_df.count()}")
print(f"Columns:   {bronze_df.columns}")
print()
bronze_df.printSchema()
print()
bronze_df.show(5, truncate=False)

StatementMeta(, a9b72d9b-b798-41e3-930d-2e90eb28b94a, 5, Finished, Available, Finished, False)

Row count: 20
Columns:   ['carrier_id', 'carrier_name', 'am_best_rating', 'admitted', '_ingestion_timestamp', '_source_file', '_batch_id', '_ingestion_method']

root
 |-- carrier_id: string (nullable = true)
 |-- carrier_name: string (nullable = true)
 |-- am_best_rating: string (nullable = true)
 |-- admitted: boolean (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- _ingestion_method: string (nullable = true)


+----------+-----------------------------------------------+--------------+--------+--------------------------+------------+-------------------------------+-----------------+
|carrier_id|carrier_name                                   |am_best_rating|admitted|_ingestion_timestamp      |_source_file|_batch_id                      |_ingestion_method|
+----------+-----------------------------------------------+--------------+--------+--------------------------+---------

In [4]:
# Promote bronze to silver: keep the domain columns as-is (types are already
# clean for this table), rename bronze _batch_id to _bronze_batch_id so it
# doesn't collide with the silver batch ID we add below, and drop the bronze
# lineage columns that silver doesn't need to carry forward.
silver_df = (
    bronze_df
    .withColumnRenamed("_batch_id", "_bronze_batch_id")
    .drop("_ingestion_timestamp", "_source_file", "_ingestion_method")
    .withColumn("_silver_load_timestamp", current_timestamp())
    .withColumn("_silver_batch_id", lit(SILVER_BATCH_ID))
)

print(f"Row count: {silver_df.count()}")
print(f"Columns:   {silver_df.columns}")
print()
silver_df.printSchema()
print()
silver_df.show(5, truncate=False)

StatementMeta(, a9b72d9b-b798-41e3-930d-2e90eb28b94a, 6, Finished, Available, Finished, False)

Row count: 20
Columns:   ['carrier_id', 'carrier_name', 'am_best_rating', 'admitted', '_bronze_batch_id', '_silver_load_timestamp', '_silver_batch_id']

root
 |-- carrier_id: string (nullable = true)
 |-- carrier_name: string (nullable = true)
 |-- am_best_rating: string (nullable = true)
 |-- admitted: boolean (nullable = true)
 |-- _bronze_batch_id: string (nullable = true)
 |-- _silver_load_timestamp: timestamp (nullable = false)
 |-- _silver_batch_id: string (nullable = false)


+----------+-----------------------------------------------+--------------+--------+-------------------------------+--------------------------+-------------------------------+
|carrier_id|carrier_name                                   |am_best_rating|admitted|_bronze_batch_id               |_silver_load_timestamp    |_silver_batch_id               |
+----------+-----------------------------------------------+--------------+--------+-------------------------------+--------------------------+-----------------

In [5]:
# Pre-write validation. Any failure halts the write.
# am_best_rating check is the domain-specific one — signals we know what
# valid carrier ratings look like in insurance.

VALID_AM_BEST_RATINGS = {"A++", "A+", "A", "A-", "B++", "B+"}

null_id_count        = silver_df.filter(col("carrier_id").isNull()).count()
null_name_count      = silver_df.filter(col("carrier_name").isNull()).count()
null_rating_count    = silver_df.filter(col("am_best_rating").isNull()).count()
null_admitted_count  = silver_df.filter(col("admitted").isNull()).count()
invalid_rating_count = silver_df.filter(
    ~col("am_best_rating").isin(list(VALID_AM_BEST_RATINGS))
).count()
total_count       = silver_df.count()
distinct_id_count = silver_df.select("carrier_id").distinct().count()
bronze_count      = bronze_df.count()

print(f"Total rows:             {total_count}")
print(f"Bronze row count:       {bronze_count}")
print(f"Null carrier_id:        {null_id_count}")
print(f"Null carrier_name:      {null_name_count}")
print(f"Null am_best_rating:    {null_rating_count}")
print(f"Null admitted:          {null_admitted_count}")
print(f"Invalid am_best_rating: {invalid_rating_count}")
print(f"Distinct carrier_id:    {distinct_id_count}")
print()

assert null_id_count == 0, \
    f"Found {null_id_count} rows with null carrier_id"
assert null_name_count == 0, \
    f"Found {null_name_count} rows with null carrier_name"
assert null_rating_count == 0, \
    f"Found {null_rating_count} rows with null am_best_rating"
assert null_admitted_count == 0, \
    f"Found {null_admitted_count} rows with null admitted"
assert invalid_rating_count == 0, (
    f"Found {invalid_rating_count} rows with invalid am_best_rating. "
    f"Allowed: {VALID_AM_BEST_RATINGS}"
)
assert distinct_id_count == total_count, (
    f"Duplicate carrier_id found: {total_count} rows, {distinct_id_count} distinct"
)
assert total_count == bronze_count, (
    f"Row count mismatch: bronze={bronze_count}, silver={total_count}"
)

print("All checks passed.")

StatementMeta(, a9b72d9b-b798-41e3-930d-2e90eb28b94a, 7, Finished, Available, Finished, False)

Total rows:             20
Bronze row count:       20
Null carrier_id:        0
Null carrier_name:      0
Null am_best_rating:    0
Null admitted:          0
Invalid am_best_rating: 0
Distinct carrier_id:    20

All checks passed.


In [7]:
# Write silver_carriers as a managed Delta table in the default lakehouse
# (silver_citadel_mga). Schema enforcement is on, mode is overwrite so
# the table fully rebuilds every run.
(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TARGET_TABLE)
)

print(f"Wrote {silver_df.count()} rows to {TARGET_TABLE}")

StatementMeta(, 185bbeac-bc1c-48b7-b882-f68b901aab1d, 9, Finished, Available, Finished, False)

Wrote 20 rows to silver_carriers


In [8]:
# Read silver_carriers back from the lakehouse and confirm the round trip.
# Mirrors the verification pattern in the bronze notebooks.
verify_df = spark.read.table(TARGET_TABLE)

row_count = verify_df.count()
col_count = len(verify_df.columns)

required_lineage = ["_bronze_batch_id", "_silver_load_timestamp", "_silver_batch_id"]
has_lineage = all(c in verify_df.columns for c in required_lineage)

print(f"silver_carriers verified:")
print(f"  Rows:         {row_count}")
print(f"  Columns:      {col_count}")
print(f"  Lineage cols: {'present' if has_lineage else 'MISSING'}")
print()
print("Sample:")
verify_df.select(
    "carrier_id", "carrier_name", "am_best_rating", "admitted",
    "_bronze_batch_id", "_silver_batch_id"
).show(5, truncate=False)

StatementMeta(, 185bbeac-bc1c-48b7-b882-f68b901aab1d, 10, Finished, Available, Finished, False)

silver_carriers verified:
  Rows:         20
  Columns:      7
  Lineage cols: present

Sample:
+----------+-----------------------------------------------+--------------+--------+-------------------------------+-------------------------------+
|carrier_id|carrier_name                                   |am_best_rating|admitted|_bronze_batch_id               |_silver_batch_id               |
+----------+-----------------------------------------------+--------------+--------+-------------------------------+-------------------------------+
|CAR-001   |Rodriguez, Figueroa and Sanchez Insurance Group|A             |false   |bronze_20260601_023424_4f334c9f|silver_20260601_201001_a2d88919|
|CAR-002   |Doyle Ltd Insurance Group                      |A-            |true    |bronze_20260601_023424_4f334c9f|silver_20260601_201001_a2d88919|
|CAR-003   |Mcclain, Miller and Henderson Casualty         |A+            |true    |bronze_20260601_023424_4f334c9f|silver_20260601_201001_a2d88919|
|CAR-004  